# Compare & Compile Niger Health Facility Source

1. [OSM extract, Humanitarian OpenStreetMap Team (HOT)](https://data.humdata.org/dataset/hotosm_ner_health_facilities)
2. [OSM extract, Global Healthsites Mapping Project](https://data.humdata.org/dataset/niger-healthsites)
3. [Google Earth Manual Review](https://earth.google.com/web/search/6.353554,2.416163/@6.95412207,3.01626942,389.70194553a,1226330.81984863d,34.99999875y,-0h,0t,0r/data=CiwiJgokCesg4VjhsDtAEXh2un9jUzXAGV999jW0ZVhAIXfgXU5pQTbAQgIIATIpCicKJQohMUF3R19pTUVwcG9EenFjMmV4VnZVbGtzdm5QeVFMTlJtIAE6AwoBMEICCABKCAiBzvbgBhAB)

In [16]:
import pandas as pd
import geopandas as gpd

from shapely.geometry import Point
from shapely.ops import unary_union

In [29]:
osm1 = gpd.read_file("ii. health-facilities/HDX/niger/hotosm_ner_health_facilities_points_geojson.geojson")
osm2 = gpd.read_file("ii. health-facilities/HDX/niger/niger.geojson")
ge = gpd.read_file("ii. health-facilities/HDX/niger/ge_manual.csv")

/Users/kt/anaconda3/envs/bigd/lib/python3.11/site-packages/geopandas/io/file.py:399: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  as_dt = pd.to_datetime(df[k], errors="ignore")


In [30]:
print(len(osm1), len(osm2), len(ge))

547 615 12


In [31]:
# Get centroids of polygons for osm2
osm2['geometry_orig'] = osm2['geometry'].to_wkt()
osm2['geometry'] = osm2.geometry.centroid


# Ensure all GeoDataFrames are using the same CRS
target_crs = "EPSG:32632"  # UTM zone 32N is good for Niger; adjust if needed
osm1 = osm1.to_crs(target_crs)
osm2 = osm2.to_crs(target_crs)

# Convert CSV to GeoDataFrame if necessary
if not isinstance(ge.geometry.iloc[0], Point):
    ge['geometry'] = gpd.points_from_xy(ge['lon'], ge['lat'])  # update with correct column names
ge = gpd.GeoDataFrame(ge, geometry='geometry', crs="EPSG:4326").to_crs(target_crs)

# Step 1: Remove duplicates within 10 meters from osm2 based on osm1
# Buffer each point in osm1 by 10 meters
osm1_buffered = osm1.copy()
osm1_buffered['geometry'] = osm1_buffered.geometry.buffer(10)

# Spatial join: keep only osm2 points that are NOT within any osm1 buffer
osm2_unique = gpd.sjoin(osm2, osm1_buffered[['geometry']], how='left', predicate='within')
osm2_unique = osm2_unique[osm2_unique['index_right'].isna()].drop(columns=['index_right'])

# Step 2: Combine all three datasets
combined = pd.concat([osm1, osm2_unique, ge], ignore_index=True)
combined = combined.drop_duplicates(subset='geometry')  # optional: drop exact duplicates

# Optional: reset CRS to WGS84
combined = combined.to_crs("EPSG:4326")

# Save result
combined.to_file("ii. health-facilities/HDX/niger/master_facilities_niger.geojson", driver="GeoJSON")

print(f"Final master file has {len(combined)} unique points.")

/var/folders/b3/qhkpbfdd3y36ys5y_kqndwbc0000gn/T/ipykernel_50526/2800569618.py:3: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  osm2['geometry'] = osm2.geometry.centroid


Final master file has 720 unique points.


Code from ii_hdx_health.py to remove irrelevant facilities:

In [32]:
import re

gdf = combined.copy()


# Pre-set exclusionary descriptions for out of scope facilities
exclusion = ['dentist', 'blood_donation', 'laboratory', 'paediatrics',
'ophthalmology', 'optometrist', 'neurologie', 'neurology', 'Neurologie Pédiatrie',
'neurologie;gynaecology;paediatrics;cardiology;biology;radiology',
            'general;gynaecology', 'cardiology', 'Infirmerie'
            
# other guesses
'endoscopy', 'radiology', 'radiologie', 'xray'
            ]

# get unique columns in dataframe
hdx_columns = [col for col in ['healthcare', 'healthcare:specialty', '#meta+healthcare', 'fac_type_orig',
                                'group_fac_type'] if col in gdf.columns]

if len(hdx_columns) > 0:
    print(f'Found columns: ', hdx_columns)

    # remove dentists, specialists...etc.
    start = len(gdf)
    for col in hdx_columns:
        gdf = gdf.loc[~gdf[col].isin(exclusion)]

    # print summary
    print (f'{start - len(gdf)} records removed.')

# if the columns were not found in the dataframe, do nothing
else:
    print("No columns found: ", gdf.columns)

# export
gdf.to_file(f'../reach/health-centres/HDX/niger/niger_hdx.geojson')

Found columns:  ['healthcare']
8 records removed.
